# PhotoWalk — Nerfstudio Training on Google Colab

This notebook trains a 3D Gaussian Splat scene using Nerfstudio Splatfacto.

**Before running:** Runtime → Change runtime type → T4 GPU

**Two modes:**
- **Mode A (default):** Uses the gerrard-hall test dataset (downloads automatically, no photos needed)
- **Mode B:** Uses your own photos uploaded to Google Drive

Run cells top to bottom. Each cell is independent and labeled.

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install Nerfstudio + COLMAP ──────────────────────────────────────
# Takes 3-5 minutes.
!apt-get install -y colmap -q
!pip install nerfstudio -q
print('Done.')

In [ ]:
# ── Cell 3: Verify install ───────────────────────────────────────────────────
!ns-train --help 2>&1 | head -3
!colmap help 2>&1 | head -3
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## Mode A — Gerrard Hall test dataset (no photos needed)
Run cells 4A through 7. Skip to Mode B if using your own photos.

In [ ]:
# ── Cell 4A: Download gerrard-hall dataset ───────────────────────────────────
import os

SCENE = 'gerrard-hall'
BASE  = f'/content/photowalk/scenes/{SCENE}'
os.makedirs(BASE, exist_ok=True)

zip_path = f'{BASE}/gerrard-hall.zip'
if not os.path.exists(zip_path):
    print('Downloading gerrard-hall (~960 MB)...')
    !wget -q --show-progress -O {zip_path} \
        https://github.com/colmap/colmap/releases/download/3.11.1/gerrard-hall.zip
    print('Download complete.')
else:
    print('Already downloaded.')

In [ ]:
# ── Cell 5A: Extract and set up paths ────────────────────────────────────────
import zipfile, os, shutil

SCENE     = 'gerrard-hall'
BASE      = f'/content/photowalk/scenes/{SCENE}'
EXTRACTED = f'{BASE}/gerrard-hall'
IMAGES    = f'{EXTRACTED}/images'
SPARSE    = f'{EXTRACTED}/sparse'
NS_DATA   = f'{BASE}/nerfstudio_data'
NS_OUT    = f'{BASE}/nerfstudio_outputs'
EXPORT    = f'{BASE}/exports/splat'

for d in [NS_DATA, NS_OUT, EXPORT]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(IMAGES):
    print('Extracting...')
    with zipfile.ZipFile(f'{BASE}/gerrard-hall.zip', 'r') as z:
        z.extractall(BASE)
    print('Extracted.')
else:
    print('Already extracted.')

# Fix: nerfstudio --skip-colmap expects sparse model in sparse/0/ subdirectory.
# gerrard-hall stores cameras.txt etc. directly in sparse/ — copy to sparse/0/.
SPARSE_0 = f'{SPARSE}/0'
if not os.path.exists(SPARSE_0):
    os.makedirs(SPARSE_0)
    for fname in ['cameras.txt', 'images.txt', 'points3D.txt']:
        src = f'{SPARSE}/{fname}'
        if os.path.exists(src):
            shutil.copy2(src, f'{SPARSE_0}/{fname}')
    print(f'Created sparse/0/ with: {os.listdir(SPARSE_0)}')
else:
    print(f'sparse/0/ already exists: {os.listdir(SPARSE_0)}')

imgs = [f for f in os.listdir(IMAGES) if f.lower().endswith('.jpg')]
print(f'Images: {len(imgs)}')

In [ ]:
# ── Cell 6A: Run ns-process-data (skip COLMAP, use existing sparse model) ────
# Takes 2-5 minutes — runs image undistortion and writes transforms.json.

SCENE   = 'gerrard-hall'
BASE    = f'/content/photowalk/scenes/{SCENE}'
IMAGES  = f'{BASE}/gerrard-hall/images'
SPARSE  = f'{BASE}/gerrard-hall/sparse'   # nerfstudio looks for sparse/0/ inside here
NS_DATA = f'{BASE}/nerfstudio_data'

!ns-process-data images \
    --data {IMAGES} \
    --output-dir {NS_DATA} \
    --skip-colmap \
    --colmap-model-path {SPARSE} \
    --verbose

In [ ]:
# ── Cell 6A-fallback: Full COLMAP from scratch (only if Cell 6A failed) ───────
# Runs feature extraction + matching + mapping. Takes 10-20 minutes.

SCENE   = 'gerrard-hall'
BASE    = f'/content/photowalk/scenes/{SCENE}'
IMAGES  = f'{BASE}/gerrard-hall/images'
NS_DATA = f'{BASE}/nerfstudio_data'

import shutil, os
shutil.rmtree(NS_DATA, ignore_errors=True)
os.makedirs(NS_DATA)

!ns-process-data images \
    --data {IMAGES} \
    --output-dir {NS_DATA} \
    --verbose

---
## Mode B — Your own photos from Google Drive
Skip this section if you ran Mode A.

In [ ]:
# ── Cell 4B: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

In [ ]:
# ── Cell 5B: Configure your scene ────────────────────────────────────────────
# Edit these two lines:
SCENE        = 'desk_scene'
DRIVE_IMAGES = '/content/drive/MyDrive/photowalk/desk_scene/processed'

import os
BASE    = f'/content/photowalk/scenes/{SCENE}'
IMAGES  = DRIVE_IMAGES
NS_DATA = f'{BASE}/nerfstudio_data'
NS_OUT  = f'{BASE}/nerfstudio_outputs'
EXPORT  = f'{BASE}/exports/splat'

for d in [NS_DATA, NS_OUT, EXPORT]:
    os.makedirs(d, exist_ok=True)

imgs = [f for f in os.listdir(IMAGES) if f.lower().endswith(('.jpg','.jpeg','.png'))]
print(f'Found {len(imgs)} images in {IMAGES}')

In [ ]:
# ── Cell 6B: Run ns-process-data (full COLMAP) ───────────────────────────────
# Takes 5-15 minutes depending on image count.

!ns-process-data images \
    --data {IMAGES} \
    --output-dir {NS_DATA} \
    --verbose

---
## Training — runs for both Mode A and Mode B

In [ ]:
# ── Cell 7: Verify transforms.json was created ───────────────────────────────
import os, json

transforms = f'{NS_DATA}/transforms.json'
if not os.path.exists(transforms):
    print('ERROR: transforms.json not found — ns-process-data did not complete.')
    print(f'Contents of {NS_DATA}:')
    print(os.listdir(NS_DATA))
else:
    with open(transforms) as f:
        t = json.load(f)
    print(f'transforms.json found — {len(t["frames"])} camera frames registered')
    print('Ready to train.')

In [ ]:
# ── Cell 8: Train Splatfacto ─────────────────────────────────────────────────
# 30,000 iterations. Takes 45-90 minutes on T4.
# To train faster add: --max-num-iterations 15000

!ns-train splatfacto \
    --data {NS_DATA} \
    --output-dir {NS_OUT} \
    --viewer.quit-on-train-completion True

In [ ]:
# ── Cell 9: Find the config.yml ──────────────────────────────────────────────
import glob, os

configs = glob.glob(f'{NS_OUT}/**/config.yml', recursive=True)
if not configs:
    print('ERROR: No config.yml found. Did training complete?')
else:
    CONFIG = max(configs, key=os.path.getmtime)
    print(f'Config found: {CONFIG}')

In [ ]:
# ── Cell 10: Export Gaussian Splat (.ply) ─────────────────────────────────────

!ns-export gaussian-splat \
    --load-config {CONFIG} \
    --output-dir {EXPORT}

import os
ply_files = [f for f in os.listdir(EXPORT) if f.endswith('.ply')]
print(f'Exported: {ply_files}')
for f in ply_files:
    size_mb = os.path.getsize(f'{EXPORT}/{f}') / 1e6
    print(f'  {f}: {size_mb:.1f} MB')

In [ ]:
# ── Cell 11: Download the .ply to your computer ───────────────────────────────
from google.colab import files
import os

ply_files = [f'{EXPORT}/{f}' for f in os.listdir(EXPORT) if f.endswith('.ply')]
if not ply_files:
    print('No .ply files found.')
else:
    for ply in ply_files:
        print(f'Downloading {ply}...')
        files.download(ply)
    print('Done. Load the .ply directly in the landing page web viewer.')

In [ ]:
# ── Cell 12 (optional): Save outputs to Google Drive ─────────────────────────
DRIVE_SAVE_PATH = '/content/drive/MyDrive/photowalk/exports'

import shutil, os
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
for f in os.listdir(EXPORT):
    src = f'{EXPORT}/{f}'
    dst = f'{DRIVE_SAVE_PATH}/{SCENE}_{f}'
    shutil.copy2(src, dst)
    print(f'Saved: {dst}')

---
## After downloading the .ply

The landing page has a built-in browser viewer (gsplat). To add your scene:

1. Host the `.ply` somewhere with public HTTPS + CORS — easiest options:
   - Upload to HuggingFace dataset repo (free, CORS enabled)
   - GitHub Releases (free for files under 2 GB)
2. Copy the direct download URL
3. Paste it into `web/src/data/scenes.js` → `splat_url` field for your scene
4. Run `cd web && npm run dev` to test locally

---
## Experiment runs (image count)

To test 20 vs 50 vs 100 images, subset before ns-process-data:

```python
import os, shutil

all_imgs = sorted(os.listdir(IMAGES))
IMAGES_20 = '/content/photowalk/scenes/gerrard-hall-20/images'
os.makedirs(IMAGES_20, exist_ok=True)
for f in all_imgs[:20]:
    shutil.copy(f'{IMAGES}/{f}', f'{IMAGES_20}/{f}')
# Then re-run ns-process-data pointing to IMAGES_20
```

---
## Troubleshooting

| Problem | Fix |
|---|---|
| `CUDA not available` | Runtime → Change runtime type → T4 GPU, reconnect |
| `ns-train: command not found` | Re-run Cell 2 |
| `colmap: command not found` | Re-run Cell 2 (`apt-get install colmap`) |
| transforms.json not found after 6A | Run Cell 6A-fallback (full COLMAP) |
| Training OOM crash | Add `--pipeline.datamanager.train-num-rays-per-batch 2048` to ns-train |
| Session timed out | Re-run from Cell 1, use Cell 12 to save to Drive first |
| 0 cameras registered | Too little overlap or texture in images |
